In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from pydantic import BaseModel, Field
from typing import Literal
from dotenv import load_dotenv

from pathlib import Path
import base64
from IPython.display import Markdown

load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [2]:
def get_image_base64(file_path):
    path = Path(file_path)
    image_bytes = path.read_bytes()
    base64_bytes = base64.b64encode(image_bytes)
    base64_string = base64_bytes.decode("utf-8")
    return base64_string

In [3]:
class Contacts(BaseModel):
    type: Literal["phone", "email", "website"]
    value: str

class HospitalInfo(BaseModel):
    name: str
    dr_name: str
    address: str
    contacts: list[Contacts]

class PatientInfo(BaseModel):
    name: str
    age: int
    gender: Literal["female", "male", "others"]

class Investigation(BaseModel):
    name: str
    result: str
    reference_val: str
    unit: str


class LabReport(BaseModel):
    hospital_info: HospitalInfo
    patient_info: PatientInfo
    investigation_type: str
    investigations: list[Investigation]

llm_with_schema = llm.with_structured_output(LabReport)

In [12]:
messages =[
    SystemMessage(
        """
        You are an expert Lab Report reader.
        You are provided with an image of a lab report.
        Extract required information from the image and return in given structure.
        CONSTRAINTS:
            - Reply "Invalid Image - <Reason>" in case if the provided image is not a Lab Report
        """
    ),
 HumanMessage(
        content=[
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{get_image_base64("./media/cat-and-spidey.jpg")}"
                },
            }
        ]
    )
]

In [13]:
ai_message = llm_with_schema.invoke(messages)

In [14]:
ai_message.model_dump()

{'hospital_info': {'name': 'Invalid Image - Not a Lab Report',
  'dr_name': 'Invalid Image - Not a Lab Report',
  'address': 'Invalid Image - Not a Lab Report',
  'contacts': [{'type': 'phone',
    'value': 'Invalid Image - Not a Lab Report'}]},
 'patient_info': {'name': 'Invalid Image - Not a Lab Report',
  'age': 0,
  'gender': 'others'},
 'investigation_type': 'Invalid Image - Not a Lab Report',
 'investigations': [{'name': 'Invalid Image - Not a Lab Report',
   'result': 'Invalid Image - Not a Lab Report',
   'reference_val': 'Invalid Image - Not a Lab Report',
   'unit': 'Invalid Image - Not a Lab Report'}]}